# 🎯 Your First LLM API Calls

**Master the fundamentals of calling Large Language Model APIs**

Learn to work with OpenAI GPT-4, Anthropic Claude, and Groq - understanding costs, parameters, and best practices.

---

## 📋 Overview

**What you'll learn:**
- Make API calls to OpenAI, Anthropic, and Groq
- Understand and control API parameters
- Calculate costs accurately
- Handle errors and rate limits
- Compare provider performance and pricing

**Prerequisites:** 
- Completed `00_setup/00_environment_setup.ipynb`
- At least one API key configured
- Basic Python knowledge

**Time estimate:** ⏱️ 45-60 minutes

**Difficulty:** 🟢 Beginner

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. ✅ Make successful API calls to all major LLM providers
2. ✅ Understand temperature, max_tokens, and other key parameters
3. ✅ Calculate exact costs for every API call
4. ✅ Implement proper error handling
5. ✅ Choose the right provider for different use cases

---

## 📖 Understanding LLM APIs

### What is an API?

**API** (Application Programming Interface) = A way for your code to talk to someone else's service

Think of it like ordering food at a restaurant:
```
You (your code) → Waiter (API) → Kitchen (LLM) → Waiter → You
```

### How LLM APIs Work

1. **You send** a request with:
   - Your API key (authentication)
   - A prompt (the question/task)
   - Parameters (temperature, max_tokens, etc.)

2. **The model processes** your request:
   - Tokenizes your text (breaks into pieces)
   - Runs through neural network
   - Generates response token by token

3. **You receive** a response with:
   - Generated text
   - Token counts (input + output)
   - Metadata (model used, etc.)

### Why Multiple Providers?

Different providers have different strengths:

| Provider | Strengths | Best For |
|----------|-----------|----------|
| **OpenAI** | Most popular, best docs | Production apps, complex tasks |
| **Anthropic** | Long context, safety-focused | Document analysis, safe apps |
| **Groq** | Extremely fast, cheap | Development, high-volume tasks |

---

In [ ]:
# Setup: Import required libraries
import os
import time
from dotenv import load_dotenv
from typing import Dict, Any
import json

# Load environment variables
load_dotenv()

print("✅ Setup complete!")
print(f"\nAPI Keys configured:")
print(f"  OpenAI:    {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"  Anthropic: {'✅' if os.getenv('ANTHROPIC_API_KEY') else '❌'}")
print(f"  Groq:      {'✅' if os.getenv('GROQ_API_KEY') else '❌'}")

---

## 🔵 OpenAI API (GPT-4, GPT-3.5)

OpenAI created ChatGPT and provides the GPT family of models.

### Key Models:
- **gpt-4** - Most capable, best reasoning ($0.03/1K input tokens)
- **gpt-4-turbo** - Faster GPT-4 ($0.01/1K input tokens)
- **gpt-3.5-turbo** - Fast and cheap ($0.0015/1K input tokens)

### When to Use:
- Complex reasoning tasks
- Production applications
- When you need the best quality

In [ ]:
from openai import OpenAI

# Initialize OpenAI client
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def call_openai(prompt: str, model: str = "gpt-3.5-turbo", **kwargs) -> Dict[str, Any]:
    """
    Call OpenAI API and return detailed response information.
    
    Args:
        prompt: The user's message/question
        model: Which OpenAI model to use
        **kwargs: Additional parameters (temperature, max_tokens, etc.)
    
    Returns:
        Dictionary with response, tokens, cost, and timing info
    """
    # Start timing
    start_time = time.time()
    
    # Make API call
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        **kwargs
    )
    
    # Calculate elapsed time
    elapsed_time = time.time() - start_time
    
    # Extract response details
    message = response.choices[0].message.content
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens
    
    # Calculate cost (pricing as of 2024)
    if "gpt-4" in model:
        cost = (input_tokens / 1000) * 0.03 + (output_tokens / 1000) * 0.06
    else:  # gpt-3.5-turbo
        cost = (input_tokens / 1000) * 0.0015 + (output_tokens / 1000) * 0.002
    
    return {
        'provider': 'OpenAI',
        'model': model,
        'response': message,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': total_tokens,
        'cost_usd': cost,
        'latency_seconds': elapsed_time,
    }

# Test OpenAI API
print("🔵 Testing OpenAI API...\n")

result = call_openai(
    prompt="Explain what a Large Language Model is in exactly 2 sentences.",
    max_tokens=100
)

print(f"Model: {result['model']}")
print(f"\nResponse:\n{result['response']}")
print(f"\n📊 Metrics:")
print(f"  Tokens: {result['input_tokens']} in + {result['output_tokens']} out = {result['total_tokens']} total")
print(f"  Cost: ${result['cost_usd']:.6f}")
print(f"  Latency: {result['latency_seconds']:.2f}s")

### 💡 Understanding Tokens

**What is a token?**
- A piece of text (word, part of word, or character)
- English: ~1 token = 4 characters or 0.75 words
- Example: "Hello world" = 2 tokens

**Why do tokens matter?**
1. **Cost** - You pay per token
2. **Limits** - Models have max token limits
3. **Speed** - More tokens = slower response

**Rule of thumb:**
```
100 tokens ≈ 75 words
1,000 tokens ≈ 750 words
1,500 tokens ≈ 1 page of text
```

**Pro tip:** Use `max_tokens` to control costs!

---

## 🟣 Anthropic API (Claude)

Anthropic created Claude, known for safety and long context windows.

### Key Models:
- **claude-3-5-sonnet** - Best balance ($0.003/1K input tokens)
- **claude-3-opus** - Most capable ($0.015/1K input tokens)
- **claude-3-haiku** - Fastest and cheapest ($0.00025/1K input tokens)

### When to Use:
- Long documents (up to 200K tokens)
- Safety-critical applications
- Detailed analysis tasks

In [ ]:
import anthropic

# Initialize Anthropic client
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

def call_anthropic(prompt: str, model: str = "claude-3-5-sonnet-20241022", **kwargs) -> Dict[str, Any]:
    """
    Call Anthropic API and return detailed response information.
    
    Args:
        prompt: The user's message/question
        model: Which Claude model to use
        **kwargs: Additional parameters (temperature, max_tokens, etc.)
    
    Returns:
        Dictionary with response, tokens, cost, and timing info
    """
    # Start timing
    start_time = time.time()
    
    # Default max_tokens if not provided (required by Anthropic)
    if 'max_tokens' not in kwargs:
        kwargs['max_tokens'] = 1024
    
    # Make API call
    response = anthropic_client.messages.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        **kwargs
    )
    
    # Calculate elapsed time
    elapsed_time = time.time() - start_time
    
    # Extract response details
    message = response.content[0].text
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    total_tokens = input_tokens + output_tokens
    
    # Calculate cost (Claude 3.5 Sonnet pricing)
    if "opus" in model:
        cost = (input_tokens / 1000) * 0.015 + (output_tokens / 1000) * 0.075
    elif "haiku" in model:
        cost = (input_tokens / 1000) * 0.00025 + (output_tokens / 1000) * 0.00125
    else:  # sonnet
        cost = (input_tokens / 1000) * 0.003 + (output_tokens / 1000) * 0.015
    
    return {
        'provider': 'Anthropic',
        'model': model,
        'response': message,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': total_tokens,
        'cost_usd': cost,
        'latency_seconds': elapsed_time,
    }

# Test Anthropic API
print("🟣 Testing Anthropic API...\n")

result = call_anthropic(
    prompt="Explain what a Large Language Model is in exactly 2 sentences.",
    max_tokens=100
)

print(f"Model: {result['model']}")
print(f"\nResponse:\n{result['response']}")
print(f"\n📊 Metrics:")
print(f"  Tokens: {result['input_tokens']} in + {result['output_tokens']} out = {result['total_tokens']} total")
print(f"  Cost: ${result['cost_usd']:.6f}")
print(f"  Latency: {result['latency_seconds']:.2f}s")

---

## 🟢 Groq API (Ultra-Fast Inference)

Groq provides incredibly fast inference with open-source models.

### Key Models:
- **mixtral-8x7b-32768** - Fast and capable (free tier!)
- **llama-3.1-70b** - Meta's Llama 3.1
- **gemma2-9b-it** - Google's Gemma

### When to Use:
- Development and testing
- High-volume, simple tasks
- Real-time applications (super fast!)
- Learning without breaking the bank

In [ ]:
from groq import Groq

# Initialize Groq client
groq_client = Groq(api_key=os.getenv('GROQ_API_KEY'))

def call_groq(prompt: str, model: str = "mixtral-8x7b-32768", **kwargs) -> Dict[str, Any]:
    """
    Call Groq API and return detailed response information.
    
    Args:
        prompt: The user's message/question
        model: Which Groq model to use
        **kwargs: Additional parameters (temperature, max_tokens, etc.)
    
    Returns:
        Dictionary with response, tokens, cost, and timing info
    """
    # Start timing
    start_time = time.time()
    
    # Make API call
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        **kwargs
    )
    
    # Calculate elapsed time
    elapsed_time = time.time() - start_time
    
    # Extract response details
    message = response.choices[0].message.content
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens
    
    # Groq pricing (very cheap, often free tier)
    cost = 0.0  # Free tier for learning!
    
    return {
        'provider': 'Groq',
        'model': model,
        'response': message,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': total_tokens,
        'cost_usd': cost,
        'latency_seconds': elapsed_time,
    }

# Test Groq API
print("🟢 Testing Groq API...\n")

result = call_groq(
    prompt="Explain what a Large Language Model is in exactly 2 sentences.",
    max_tokens=100
)

print(f"Model: {result['model']}")
print(f"\nResponse:\n{result['response']}")
print(f"\n📊 Metrics:")
print(f"  Tokens: {result['input_tokens']} in + {result['output_tokens']} out = {result['total_tokens']} total")
print(f"  Cost: ${result['cost_usd']:.6f} (free tier!)")
print(f"  Latency: {result['latency_seconds']:.2f}s ⚡ (notice how fast!)")

### 💡 Did You Notice the Speed?

Groq is typically **3-10x faster** than other providers!

**Typical latencies:**
- Groq: 0.2-0.5 seconds 🚀
- OpenAI: 1-3 seconds
- Anthropic: 1-2 seconds

**Why is Groq so fast?**
- Custom AI chips (LPUs)
- Optimized inference
- Smaller models (but still capable!)

**When speed matters:**
- Real-time chat applications
- High-volume processing
- Interactive demos
- Development and testing

---

## 📊 Side-by-Side Comparison

Let's compare all three providers on the same task!

In [ ]:
import pandas as pd

# Same prompt for all providers
test_prompt = "Write a haiku about artificial intelligence."

print("🔬 Running comparison across all providers...\n")
print(f"Prompt: \"{test_prompt}\"\n")
print("=" * 80)

results = []

# Test each provider
providers = [
    ('OpenAI', call_openai, {'max_tokens': 100}),
    ('Anthropic', call_anthropic, {'max_tokens': 100}),
    ('Groq', call_groq, {'max_tokens': 100}),
]

for provider_name, call_func, kwargs in providers:
    try:
        result = call_func(test_prompt, **kwargs)
        results.append(result)
        
        print(f"\n{provider_name} ({result['model']}):")
        print(f"\n{result['response']}\n")
        print(f"Tokens: {result['total_tokens']} | Cost: ${result['cost_usd']:.6f} | Time: {result['latency_seconds']:.2f}s")
        print("-" * 80)
    except Exception as e:
        print(f"\n{provider_name}: ❌ Error: {str(e)[:100]}")
        print("-" * 80)

# Create comparison table
if results:
    print("\n📊 Comparison Table:\n")
    df = pd.DataFrame([{
        'Provider': r['provider'],
        'Model': r['model'].split('-')[0] + '...',  # Shorten for display
        'Tokens': r['total_tokens'],
        'Cost ($)': f"{r['cost_usd']:.6f}",
        'Latency (s)': f"{r['latency_seconds']:.2f}",
        'Response Length': len(r['response'])
    } for r in results])
    
    print(df.to_string(index=False))
    print("\n" + "=" * 80)

### 📈 Visualize the Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if results and len(results) >= 2:
    # Create subplots
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    providers = [r['provider'] for r in results]
    
    # Plot 1: Latency
    latencies = [r['latency_seconds'] for r in results]
    axes[0].bar(providers, latencies, color=['#00a67e', '#7c3aed', '#f59e0b'])
    axes[0].set_title('Response Latency (Lower is Better)', fontweight='bold')
    axes[0].set_ylabel('Seconds')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Plot 2: Cost
    costs = [r['cost_usd'] * 1000 for r in results]  # Convert to millidollars for visibility
    axes[1].bar(providers, costs, color=['#00a67e', '#7c3aed', '#f59e0b'])
    axes[1].set_title('Cost per Request (Lower is Better)', fontweight='bold')
    axes[1].set_ylabel('Cost (millidollars)')
    axes[1].grid(axis='y', alpha=0.3)
    
    # Plot 3: Tokens
    tokens = [r['total_tokens'] for r in results]
    axes[2].bar(providers, tokens, color=['#00a67e', '#7c3aed', '#f59e0b'])
    axes[2].set_title('Total Tokens Used', fontweight='bold')
    axes[2].set_ylabel('Tokens')
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Insights:")
    print(f"  • Fastest: {min(results, key=lambda x: x['latency_seconds'])['provider']}")
    print(f"  • Cheapest: {min(results, key=lambda x: x['cost_usd'])['provider']}")
    print(f"  • Most tokens: {max(results, key=lambda x: x['total_tokens'])['provider']}")
else:
    print("⚠️ Not enough results to visualize. Make sure at least 2 providers are configured.")

---

## 🎛️ Understanding API Parameters

Let's explore the most important parameters you can control.

### 1. Temperature (Creativity Control)

**What it does:** Controls randomness/creativity

**Range:** 0.0 to 2.0 (typically use 0.0-1.0)

**Effect:**
- `temperature=0.0` → Deterministic, same output every time
- `temperature=0.3` → Slightly varied, but consistent
- `temperature=0.7` → Balanced creativity (default)
- `temperature=1.0+` → Very creative, unpredictable

**Use cases:**
```python
temperature=0.0  # Code generation, factual answers, classification
temperature=0.3  # Customer support, documentation
temperature=0.7  # General chatbot, content creation
temperature=1.0  # Creative writing, brainstorming
```

In [ ]:
# Demonstrate temperature effect
prompt = "Complete this sentence: The secret to happiness is"

print("🌡️ Temperature Demonstration\n")
print(f"Prompt: \"{prompt}\"\n")
print("=" * 80)

temperatures = [0.0, 0.5, 1.0, 1.5]

for temp in temperatures:
    result = call_groq(prompt, temperature=temp, max_tokens=30)
    print(f"\nTemperature = {temp}:")
    print(f"{result['response']}")
    print("-" * 80)

print("\n💡 Notice how higher temperatures produce more varied and creative responses!")

### 2. max_tokens (Response Length Control)

**What it does:** Limits the length of the response

**Why it matters:**
1. **Cost control** - Prevents runaway costs
2. **Latency** - Shorter responses = faster
3. **Focused answers** - Forces conciseness

**Guidelines:**
```python
max_tokens=50     # Short answer, 1-2 sentences
max_tokens=150    # Medium answer, paragraph
max_tokens=500    # Long answer, detailed explanation
max_tokens=2000   # Very long, comprehensive response
```

**⚠️ IMPORTANT:** Always set max_tokens in production!

In [ ]:
# Demonstrate max_tokens effect
prompt = "Explain machine learning."

print("📏 max_tokens Demonstration\n")
print(f"Prompt: \"{prompt}\"\n")
print("=" * 80)

token_limits = [20, 50, 100, 200]

for limit in token_limits:
    result = call_groq(prompt, max_tokens=limit, temperature=0.3)
    print(f"\nmax_tokens = {limit}:")
    print(f"{result['response']}")
    print(f"Actual tokens used: {result['output_tokens']} | Cost: ${result['cost_usd']:.6f}")
    print("-" * 80)

print("\n💡 Notice how limiting tokens forces more concise responses and reduces cost!")

### 3. top_p (Nucleus Sampling)

**What it does:** Alternative to temperature for controlling randomness

**Range:** 0.0 to 1.0

**How it works:**
- Considers only the most probable tokens that sum to top_p probability
- `top_p=0.1` → Very focused, only top 10% most likely tokens
- `top_p=0.9` → Balanced (default)
- `top_p=1.0` → All tokens considered

**Pro tip:** Use either temperature OR top_p, not both!

---

## 💰 Cost Calculator

Let's build a comprehensive cost calculator!

In [ ]:
class LLMCostCalculator:
    """
    Calculate costs for different LLM providers and models.
    Prices as of 2024 - check provider websites for current pricing.
    """
    
    # Pricing per 1K tokens (input, output)
    PRICING = {
        'openai': {
            'gpt-4': (0.03, 0.06),
            'gpt-4-turbo': (0.01, 0.03),
            'gpt-3.5-turbo': (0.0015, 0.002),
        },
        'anthropic': {
            'claude-3-opus': (0.015, 0.075),
            'claude-3-sonnet': (0.003, 0.015),
            'claude-3-haiku': (0.00025, 0.00125),
        },
        'groq': {
            'mixtral': (0.0, 0.0),  # Free tier
            'llama': (0.0, 0.0),    # Free tier
        }
    }
    
    @staticmethod
    def calculate_cost(provider: str, model: str, input_tokens: int, output_tokens: int) -> float:
        """
        Calculate cost for a specific API call.
        
        Args:
            provider: 'openai', 'anthropic', or 'groq'
            model: Model identifier
            input_tokens: Number of input tokens
            output_tokens: Number of output tokens
        
        Returns:
            Cost in USD
        """
        # Normalize model name
        model_key = model.lower()
        for key in LLMCostCalculator.PRICING.get(provider, {}).keys():
            if key in model_key:
                input_price, output_price = LLMCostCalculator.PRICING[provider][key]
                cost = (input_tokens / 1000) * input_price + (output_tokens / 1000) * output_price
                return cost
        return 0.0
    
    @staticmethod
    def estimate_monthly_cost(calls_per_day: int, avg_input_tokens: int, avg_output_tokens: int, 
                             provider: str, model: str) -> Dict[str, float]:
        """
        Estimate monthly costs based on usage patterns.
        """
        cost_per_call = LLMCostCalculator.calculate_cost(provider, model, avg_input_tokens, avg_output_tokens)
        daily_cost = cost_per_call * calls_per_day
        monthly_cost = daily_cost * 30
        
        return {
            'cost_per_call': cost_per_call,
            'daily_cost': daily_cost,
            'monthly_cost': monthly_cost,
            'yearly_cost': monthly_cost * 12
        }

# Test the calculator
print("💰 Cost Calculator Demo\n")
print("=" * 80)

# Example: Customer support chatbot
scenario = {
    'name': 'Customer Support Chatbot',
    'calls_per_day': 1000,
    'avg_input_tokens': 100,  # Customer question + context
    'avg_output_tokens': 200,  # Assistant response
}

print(f"Scenario: {scenario['name']}")
print(f"Usage: {scenario['calls_per_day']} calls/day")
print(f"Average tokens: {scenario['avg_input_tokens']} in, {scenario['avg_output_tokens']} out\n")

# Compare costs across providers
comparisons = [
    ('openai', 'gpt-3.5-turbo'),
    ('openai', 'gpt-4'),
    ('anthropic', 'claude-3-haiku'),
    ('anthropic', 'claude-3-sonnet'),
    ('groq', 'mixtral'),
]

for provider, model in comparisons:
    costs = LLMCostCalculator.estimate_monthly_cost(
        scenario['calls_per_day'],
        scenario['avg_input_tokens'],
        scenario['avg_output_tokens'],
        provider,
        model
    )
    
    print(f"{provider.title()} - {model}:")
    print(f"  Per call: ${costs['cost_per_call']:.6f}")
    print(f"  Daily: ${costs['daily_cost']:.2f}")
    print(f"  Monthly: ${costs['monthly_cost']:.2f}")
    print(f"  Yearly: ${costs['yearly_cost']:.2f}")
    print()

print("=" * 80)
print("\n💡 Key Takeaway: Model choice has HUGE cost implications!")
print("   • Groq: Perfect for development, saves $$$")
print("   • Claude Haiku: Cheap and fast for production")
print("   • GPT-3.5: Good balance for general use")
print("   • GPT-4/Claude Opus: Use only when quality is critical")

---

## ⚠️ Error Handling

Production code needs robust error handling!

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10)
)
def call_llm_with_retry(provider: str, prompt: str, **kwargs) -> Dict[str, Any]:
    """
    Call LLM API with automatic retry on failure.
    
    Handles:
    - Rate limit errors (wait and retry)
    - Network errors (retry)
    - Invalid API keys (no retry)
    - Timeout errors (retry)
    """
    try:
        if provider == 'openai':
            return call_openai(prompt, **kwargs)
        elif provider == 'anthropic':
            return call_anthropic(prompt, **kwargs)
        elif provider == 'groq':
            return call_groq(prompt, **kwargs)
        else:
            raise ValueError(f"Unknown provider: {provider}")
            
    except Exception as e:
        error_msg = str(e).lower()
        
        # Authentication errors - don't retry
        if 'authentication' in error_msg or 'api key' in error_msg:
            logger.error(f"❌ Authentication failed for {provider}: {e}")
            raise
        
        # Rate limit - will retry with backoff
        if 'rate limit' in error_msg:
            logger.warning(f"⚠️ Rate limit hit for {provider}, retrying...")
            raise
        
        # Network errors - will retry
        if 'connection' in error_msg or 'timeout' in error_msg:
            logger.warning(f"⚠️ Network error for {provider}, retrying...")
            raise
        
        # Other errors
        logger.error(f"❌ Error calling {provider}: {e}")
        raise

# Test error handling
print("🛡️ Testing Error Handling\n")

try:
    result = call_llm_with_retry(
        'groq',
        "What is Python?",
        max_tokens=50
    )
    print(f"✅ Success: {result['response'][:100]}...")
except Exception as e:
    print(f"❌ Failed after retries: {e}")

print("\n💡 In production, always implement:")
print("   • Retry logic with exponential backoff")
print("   • Proper error logging")
print("   • Fallback to alternative providers")
print("   • User-friendly error messages")

---

## 🎯 Hands-On Exercise

### Challenge: Build a Smart Router

**Task:** Create a function that automatically routes queries to the best provider based on requirements.

**Requirements:**
1. Simple queries → Groq (fast and free)
2. Complex reasoning → GPT-4 (best quality)
3. Long documents → Claude (best context window)
4. Calculate and log costs
5. Handle errors gracefully

**Starter code:**

In [ ]:
def smart_llm_router(prompt: str, complexity: str = 'simple', **kwargs) -> Dict[str, Any]:
    """
    Route LLM queries to the best provider based on requirements.
    
    Args:
        prompt: The user's query
        complexity: 'simple', 'medium', 'complex'
        **kwargs: Additional parameters
    
    Returns:
        Response dictionary with provider info
    """
    # TODO: Implement routing logic
    # Hint: Check complexity and document length
    # Hint: Use try/except for fallbacks
    pass

# Test cases
test_cases = [
    ("What is 2+2?", "simple"),
    ("Explain quantum computing and its applications.", "medium"),
    ("Analyze the philosophical implications of consciousness in AI systems.", "complex"),
]

for prompt, complexity in test_cases:
    result = smart_llm_router(prompt, complexity)
    print(f"Complexity: {complexity} → Provider: {result.get('provider', 'N/A')}")

### 💡 Solution

In [ ]:
def smart_llm_router(prompt: str, complexity: str = 'simple', **kwargs) -> Dict[str, Any]:
    """
    Route LLM queries to the best provider based on requirements.
    """
    # Determine provider based on complexity
    if complexity == 'simple':
        # Use Groq for simple queries (fast and free)
        try:
            result = call_groq(prompt, **kwargs)
            result['routing_reason'] = 'Simple query → Groq (fast & free)'
            return result
        except:
            # Fallback to GPT-3.5
            result = call_openai(prompt, model='gpt-3.5-turbo', **kwargs)
            result['routing_reason'] = 'Fallback to GPT-3.5'
            return result
    
    elif complexity == 'medium':
        # Use Claude Sonnet for balanced performance
        try:
            result = call_anthropic(prompt, model='claude-3-5-sonnet-20241022', **kwargs)
            result['routing_reason'] = 'Medium complexity → Claude Sonnet'
            return result
        except:
            # Fallback to Groq
            result = call_groq(prompt, **kwargs)
            result['routing_reason'] = 'Fallback to Groq'
            return result
    
    else:  # complex
        # Use GPT-4 for complex reasoning
        try:
            result = call_openai(prompt, model='gpt-4', **kwargs)
            result['routing_reason'] = 'Complex query → GPT-4 (best quality)'
            return result
        except:
            # Fallback to Claude
            result = call_anthropic(prompt, **kwargs)
            result['routing_reason'] = 'Fallback to Claude'
            return result

# Test the router
print("🎯 Smart Router Demo\n")
print("=" * 80)

test_cases = [
    ("What is 2+2?", "simple"),
    ("Explain quantum computing.", "medium"),
]

for prompt, complexity in test_cases:
    result = smart_llm_router(prompt, complexity, max_tokens=100)
    print(f"\nPrompt: {prompt}")
    print(f"Routing: {result['routing_reason']}")
    print(f"Cost: ${result['cost_usd']:.6f} | Latency: {result['latency_seconds']:.2f}s")
    print("-" * 80)

---

## ⚠️ Common Pitfalls

### 1. Not Setting max_tokens
```python
# ❌ WRONG - Can lead to huge costs!
response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": prompt}]
)

# ✅ RIGHT - Always set a limit
response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=500  # Set appropriate limit
)
```

### 2. Hardcoding API Keys
```python
# ❌ NEVER DO THIS!
client = OpenAI(api_key="sk-1234567890abcdef")

# ✅ Always use environment variables
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
```

### 3. No Error Handling
```python
# ❌ Will crash on any error
response = call_openai(prompt)

# ✅ Handle errors gracefully
try:
    response = call_openai(prompt)
except Exception as e:
    logger.error(f"API call failed: {e}")
    # Fallback logic
```

### 4. Ignoring Rate Limits
```python
# ❌ Will hit rate limits quickly
for prompt in many_prompts:
    response = call_api(prompt)

# ✅ Add rate limiting
import time
for prompt in many_prompts:
    response = call_api(prompt)
    time.sleep(0.1)  # Or use a proper rate limiter
```

### 5. Not Monitoring Costs
```python
# ❌ No cost tracking
response = call_api(prompt)

# ✅ Track every call
response = call_api(prompt)
total_cost += response['cost_usd']
logger.info(f"Cost: ${response['cost_usd']:.6f}, Total: ${total_cost:.2f}")
```

---

## 🏭 Production Considerations

### 1. Cost Management
- Set monthly budgets
- Implement per-user rate limits
- Cache responses when possible
- Use cheaper models for simple tasks

### 2. Reliability
- Implement retry logic with exponential backoff
- Have fallback providers
- Monitor API status pages
- Set appropriate timeouts

### 3. Monitoring
- Log all API calls
- Track costs in real-time
- Monitor latency and errors
- Set up alerts for anomalies

### 4. Security
- Rotate API keys regularly
- Use API key management services
- Implement rate limiting
- Validate and sanitize inputs

### 5. Optimization
- Batch requests when possible
- Use streaming for long responses
- Implement semantic caching
- Choose appropriate models per task

We'll learn all these techniques in upcoming notebooks!

---

## 📚 Further Reading

### Official Documentation
- [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
- [Anthropic Claude API](https://docs.anthropic.com/claude/reference)
- [Groq Documentation](https://console.groq.com/docs/quickstart)

### Best Practices
- [OpenAI Best Practices](https://platform.openai.com/docs/guides/production-best-practices)
- [Rate Limiting Strategies](https://platform.openai.com/docs/guides/rate-limits)
- [Cost Optimization Tips](https://help.openai.com/en/articles/6614209-how-do-i-optimize-my-costs)

### Tools
- [tiktoken](https://github.com/openai/tiktoken) - Token counting
- [tenacity](https://github.com/jd/tenacity) - Retry logic
- [OpenAI Cookbook](https://cookbook.openai.com/) - Code examples

### Next Notebooks
- `02_llm_basics/02_api_parameters.ipynb` - Deep dive into all parameters
- `02_llm_basics/03_streaming_responses.ipynb` - Real-time streaming
- `02_llm_basics/05_error_handling.ipynb` - Production-grade error handling

---

## ✅ Summary

### What You Learned

1. ✅ **API Basics**
   - Made calls to OpenAI, Anthropic, and Groq
   - Understood request/response structure
   - Compared provider performance

2. ✅ **Parameters**
   - `temperature` - Controls creativity
   - `max_tokens` - Limits response length
   - `top_p` - Alternative sampling method

3. ✅ **Cost Management**
   - Calculated exact costs per call
   - Estimated monthly costs
   - Built a cost calculator

4. ✅ **Error Handling**
   - Implemented retry logic
   - Handled rate limits
   - Created fallback strategies

5. ✅ **Best Practices**
   - Always set max_tokens
   - Use environment variables
   - Monitor costs
   - Choose right provider for task

### Key Takeaways

💡 **Different providers for different needs**
- Groq: Fast development
- OpenAI: Production quality
- Anthropic: Long context

💰 **Cost varies dramatically** - Choose wisely!

🎛️ **Parameters matter** - temperature, max_tokens, top_p

🛡️ **Always handle errors** - Networks fail, APIs have limits

📊 **Track everything** - Costs, latency, errors

### Practice Suggestions

1. Make 10 different API calls with varying parameters
2. Build a cost tracking spreadsheet for your usage
3. Implement your own smart router with custom logic
4. Try breaking things (invalid keys, rate limits) to see error handling

### Next Steps

Ready for more? Choose your path:

1. **Continue with LLM Basics:**
   - `02_llm_basics/02_api_parameters.ipynb` - Master all parameters
   - `02_llm_basics/03_streaming_responses.ipynb` - Real-time streaming

2. **Jump to Prompt Engineering:**
   - `03_prompt_engineering/01_prompt_patterns.ipynb`

3. **Or explore:**
   - `02_llm_basics/04_cost_calculation.ipynb` - Advanced cost management

---

## 🎉 Congratulations!

You can now confidently:
- Call any major LLM API
- Control parameters effectively
- Calculate and manage costs
- Handle errors like a pro
- Choose the right provider

**You're building real AI applications now!** 🚀

---

*Questions? Review the common pitfalls section or check the documentation links above.*